# 1. What you'll learn

        - fit biases and latent factors by SGD
- compare with a bias-only recommender
- trigger cold-start failure deliberately

        **The one question this notebook answers:** How can user and item factors explain sparse ratings, and what happens for someone with no history?

# 2. Setup

This lesson keeps matrix factorization for recommendations small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The notebook uses cached MovieLens 100k ratings when available, otherwise a clearly announced low-rank synthetic fallback. Ratings predict a user's held-out preference for an item.

**Small example:** With user factor [1,.5] and item factor [.8,-.2], their interaction is 1×.8+.5×(-.2)=.7; add global mean 3.5 and biases +.2,-.1 to predict 4.3.

In [ ]:
path=Path.cwd().parent/"_shared_data"/"ml-100k"/"u.data"
if path.exists():
 frame=pd.read_csv(path,sep="\t",names=["user","item","rating","timestamp"]).drop(columns="timestamp");frame=frame.iloc[:50000].copy();source="MovieLens 100k cached (first 50,000 interactions)"
else:
 print("OFFLINE BANNER: MovieLens cache absent; using seeded known low-rank ratings.");rng=np.random.default_rng(SEED);U0=rng.normal(0,.45,(180,6));V0=rng.normal(0,.45,(260,6));rows=[]
 for _ in range(18000):
  u=rng.integers(180);i=rng.integers(260);r=np.clip(np.round(3.4+U0[u]@V0[i]+rng.normal(0,.55)),1,5);rows.append((u+1,i+1,r))
 frame=pd.DataFrame(rows,columns=["user","item","rating"]);source="synthetic fallback"
frame["u"]=pd.Categorical(frame.user).codes;frame["i"]=pd.Categorical(frame.item).codes;print("Source:",source,"shape:",frame.shape,"users/items:",frame.u.nunique(),frame.i.nunique());print(frame.rating.value_counts(normalize=True).sort_index().round(3).to_dict())

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));frame.rating.value_counts().sort_index().plot.bar(ax=axes[0],color="#176b66");axes[0].set_title("Rating distribution")
frame.groupby("u").size().hist(bins=30,ax=axes[1],color="#d38b45");axes[1].set_title("Ratings per user")
frame.groupby("i").size().hist(bins=30,ax=axes[2],color="#768f8b");axes[2].set_title("Ratings per item");plt.tight_layout();plt.show()

**Takeaway:** Ratings are bounded and not uniformly distributed.

**Takeaway:** User history is unequal, so personal factors have unequal certainty.

**Takeaway:** Item popularity is long-tailed and creates a strong bias baseline.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
train,test=train_test_split(frame,test_size=.2,random_state=SEED);n_users=frame.u.max()+1;n_items=frame.i.max()+1;global_mean=train.rating.mean();user_mean=train.groupby("u").rating.mean();item_mean=train.groupby("i").rating.mean();print("Train/test:",train.shape,test.shape,"global mean:",round(global_mean,3))

# 6. Train

        Training turns the assumptions behind matrix factorization for recommendations into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Initialize small user/item factor vectors and zero biases.
2. Predict global mean plus biases plus factor dot product.
3. Compute each observed rating error.
4. Move only that user and item parameters down regularized squared error.

In [ ]:
rng=np.random.default_rng(SEED);k=10;P=rng.normal(0,.08,(n_users,k));Q=rng.normal(0,.08,(n_items,k));bu=np.zeros(n_users);bi=np.zeros(n_items);lr=.012;reg=.035;history=[]
rows=train[["u","i","rating"]].to_numpy()
for epoch in range(10):
 for idx in rng.permutation(len(rows)):
  u,i,r=rows[idx];u=int(u);i=int(i);err=r-(global_mean+bu[u]+bi[i]+P[u]@Q[i]);pu=P[u].copy();bu[u]+=lr*(err-reg*bu[u]);bi[i]+=lr*(err-reg*bi[i]);P[u]+=lr*(err*Q[i]-reg*P[u]);Q[i]+=lr*(err*pu-reg*Q[i])
 pred=np.clip(global_mean+bu[train.u]+bi[train.i]+np.sum(P[train.u]*Q[train.i],1),1,5);history.append(mean_squared_error(train.rating,pred)**.5)
print("Training RMSE:",[round(x,3) for x in history])

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
pred=np.clip(global_mean+bu[test.u]+bi[test.i]+np.sum(P[test.u]*Q[test.i],1),1,5);bias=np.clip(global_mean+bu[test.u]+bi[test.i],1,5);global_pred=np.repeat(global_mean,len(test));print(f"MF RMSE={mean_squared_error(test.rating,pred)**.5:.3f}; bias-only={mean_squared_error(test.rating,bias)**.5:.3f}; global={mean_squared_error(test.rating,global_pred)**.5:.3f}")
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(range(1,11),history,marker="o");axes[0].set(title="SGD learning curve",xlabel="epoch",ylabel="train RMSE")
axes[1].hexbin(test.rating,pred,gridsize=12,cmap="mako");axes[1].plot([1,5],[1,5],"--",color="white");axes[1].set(title="Held-out predictions",xlabel="rating",ylabel="prediction")
axes[2].scatter(P[:,0],P[:,1],s=8,alpha=.5);axes[2].set_title("First two user factors");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: A new user or item has no learned factor, so the model falls back to global/popularity bias and cannot personalize. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
new_user_prediction=global_mean+bi;known_user=int(train.u.value_counts().index[0]);known_prediction=global_mean+bu[known_user]+bi+Q@P[known_user];print("New-user factors do not exist; fallback spread:",round(new_user_prediction.std(),3));print("Known-user personalized spread:",round(known_prediction.std(),3));print("Cold-start recommendation is popularity/bias, not personalization.")

**Use this when…** many users and items have overlapping interaction histories and latent taste explains ratings.

        **Don't use this when…** most users/items are new, feedback is exposure-biased, or ranking and business constraints matter more than rating RMSE.

        ## Try this

        1. Change latent dimension from 2 to 30 and compare test RMSE.
2. Hold out each user's last rating instead of a random interaction.
3. Recommend top items after excluding already-rated ones.